<a href="https://colab.research.google.com/github/KijoSal-dev/Retrieval-Augmented-Generation-RAG-System-for-Question-Answering/blob/main/Salome_Kungu_CS_DA03_26054_WK12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install required libraries

In [ ]:

!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf
import re
import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# STEP 1: DEFINE CLEANING FUNCTION
# Fixes common PDF extraction errors like 'C y b e r' -> 'Cyber'

def fix_pdf_text(text):

    # Fix spacing issues from PDF extraction
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)

    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# STEP 2: Upload the PDF Document
print("Upload your PDF document")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

Upload your PDF document


Saving 123freebook.the-slipper-point-mystery-by-augusta-huiell-seaman.pdf to 123freebook.the-slipper-point-mystery-by-augusta-huiell-seaman.pdf
Uploaded file: 123freebook.the-slipper-point-mystery-by-augusta-huiell-seaman.pdf


In [ ]:
# STEP 3: Load and Process the PDF
loader = PyPDFLoader(pdf_path)

pages = loader.load()

print("Number of pages loaded:", len(pages))

# Clean the text
for page in pages:
    page.page_content = fix_pdf_text(page.page_content)

Number of pages loaded: 124


In [ ]:
# STEP 4: Split the Document into Chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print("Number of chunks created:", len(chunks))

Number of chunks created: 393


In [ ]:
# STEP 5: Create Embeddings and FAISS Vector Store
print("Creating embeddings and vector store...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("Vector store created successfully")

Creating embeddings and vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully


In [ ]:
# STEP 6: Load the Generative AI Model (FLAN-T5)
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/flan-t5-large"

print("Loading model:", model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

print("Model loaded on:", device)

Loading model: google/flan-t5-large


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on: cpu


In [ ]:
# STEP 7: Create the RAG Query Function
def query_rag(question):

    # Retrieve relevant chunks
    relevant_docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Prompt for the LLM
    input_text = f"answer the question based on the context below.\n\nQuestion: {question}\n\nContext: {context}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [ ]:
# STEP 8: Test the RAG System
print("\n----- RAG OUTPUT -----\n")

query = "What is the main setting of the story?"

result = query_rag(query)

print(result)


----- RAG OUTPUT -----

cave


In [ ]:
# STEP 9: Test with Another Question
query = "Describe the main setting of the story and its atmosphere"

print(query_rag(query))

old-fashioned farmhouse


In [ ]:
# STEP 10: Test with Another Question
query = "What significant events happen in the first few chapters?"

print(query_rag(query))

CHAPTER XII LIGHT DAWNS ON MISS CAMILLA


In [ ]:
# STEP 11: Test with Another Question
query = "Are there any recurring themes or significant symbols mentioned early on?"

print(query_rag(query))

WORD FROM THE PAST


In [3]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Salome Kungu_CS-DA03-26054_WK12.ipynb"

# Load the notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Remove the widgets metadata that's causing issues
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Save the cleaned notebook
with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Notebook cleaned! Widgets metadata removed.")

Notebook cleaned! Widgets metadata removed.


In [1]:
import os

drive_path = '/content/drive/MyDrive/Colab Notebooks '
if os.path.exists(drive_path):
    print(f"Contents of {drive_path}:")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Directory not found: {drive_path}")

Directory not found: /content/drive/MyDrive/Colab Notebooks 


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
